# S09 · Observabilidad del pipeline de viento + estimación de costos
**Nayder Arce Mayta** · Prometheus `localhost:49090` · Grafana `localhost:43000` (tablero *Viento Juliaca — Pipeline U2*)

Requisitos: `kafka/`, `uso-viento/` (productor + `consumer_viento.py`), `obs/` arriba.

In [1]:
import os, time, json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

BASE = os.getenv("VIENTO_BASE", "/opt")
ART = f"{BASE}/artifacts"
KAFKA = os.getenv("KAFKA_BOOTSTRAP", "kafka:9092")
TOPIC = "viento-eventos"
PUSHGATEWAY = os.getenv("PUSHGATEWAY", "http://pushgateway:9091")

spark = (SparkSession.builder
         .appName("s09-observabilidad-viento-nayder")
         .master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "|", time.strftime("%Y-%m-%d %H:%M:%S"))

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf77495f-667e-42af-ad97-5d7c2c7d97bd;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark 4.2.0 | 2026-11-03 01:10:54


In [2]:
# Contrato viento.lectura v1 (docs/contratos.md)
esquema_viento = (StructType()
    .add("tipoEvento", StringType()).add("version", IntegerType())
    .add("sensorId", StringType()).add("estacion", StringType())
    .add("lat", DoubleType()).add("lon", DoubleType())
    .add("velocidadViento", DoubleType()).add("direccionViento", DoubleType()).add("rafaga", DoubleType())
    .add("temperatura", DoubleType()).add("humedad", DoubleType()).add("presion", DoubleType())
    .add("obsTime", LongType()).add("origen", StringType()).add("timestamp", LongType()))

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .option("failOnDataLoss", "false")
         .load())

eventos = (crudo.select("partition", "offset", F.col("timestamp").alias("kafkaTs"),
                        F.col("value").cast("string").alias("json_str"))
           .select("partition", "offset", "kafkaTs", "json_str",
                   F.from_json("json_str", esquema_viento).alias("e"))
           .select("partition", "offset", "kafkaTs", "json_str", "e.*")
           .withColumn("ts", (F.col("timestamp") / 1000).cast("timestamp"))       # tiempo del evento (lectura)
           .withColumn("obs_ts", (F.col("obsTime") / 1000).cast("timestamp")))    # paso de 15 min del modelo
eventos.printSchema()

root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTs: timestamp (nullable = true)
 |-- json_str: string (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- version: integer (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- estacion: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocidadViento: double (nullable = true)
 |-- direccionViento: double (nullable = true)
 |-- rafaga: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- obsTime: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- obs_ts: timestamp (nullable = true)



In [3]:
import requests

def push(job, instance, metricas, metodo="PUT"):
    """Publica un grupo de métricas en el Pushgateway (formato texto de Prometheus).
    PUT reemplaza el grupo completo (job, instance)."""
    cuerpo = "".join(f"{k} {float(v)}\n" for k, v in metricas.items() if v is not None)
    url = f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instance}"
    r = requests.request(metodo, url, data=cuerpo, timeout=5)
    r.raise_for_status()
    return r.status_code

def borrar_grupo(job, instance):
    return requests.delete(f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instance}", timeout=5).status_code

## 1. Campos de observabilidad: `isValid`, `processedAt`, `latencyMs`

In [4]:
observables = (eventos
    .withColumn("isValid", F.col("tipoEvento").isNotNull() & F.col("sensorId").isNotNull()
                & F.col("velocidadViento").between(0, 200) & F.col("rafaga").between(0, 250)
                & F.col("timestamp").isNotNull())
    .withColumn("processedAt", F.unix_millis(F.current_timestamp()))
    .withColumn("latencyMs", F.col("processedAt") - F.col("timestamp"))     # publicación → procesado
    .withColumn("edadDatoMs", F.col("processedAt") - F.col("obsTime")))     # frescura del dato del modelo
observables.printSchema()

root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTs: timestamp (nullable = true)
 |-- json_str: string (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- version: integer (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- estacion: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocidadViento: double (nullable = true)
 |-- direccionViento: double (nullable = true)
 |-- rafaga: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- obsTime: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- obs_ts: timestamp (nullable = true)
 |-- isValid: boolean (nullable = true)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)
 |-- edadDatoMs: long (nullable = tru

## 2. Stream a Parquet + métricas por micro-lote al Pushgateway (`foreachBatch`)

In [5]:
P_OBS = f"{ART}/viento_eventos_observabilidad"
CHK_OBS = f"{ART}/chk_s09_obs"

def por_lote(df, batch_id):
    df = df.drop("json_str").cache()
    n = df.count()
    df.write.mode("append").parquet(P_OBS)
    if n > 0:
        r = df.agg(F.avg("latencyMs").alias("lat"), F.max("latencyMs").alias("latmax"),
                   F.avg(F.col("isValid").cast("int")).alias("ok"), F.avg("edadDatoMs").alias("edad")).first()
        push("viento_stream", "spark_s09", {
            "viento_stream_eventos_lote": n,
            "viento_stream_latencia_promedio_ms": r["lat"],
            "viento_stream_latencia_max_ms": r["latmax"],
            "viento_stream_validos_ratio": r["ok"],
            "viento_stream_edad_dato_ms": r["edad"],
            "viento_stream_batch_id": batch_id,
        })
    df.unpersist()

consulta = (observables.writeStream.foreachBatch(por_lote)
            .option("checkpointLocation", CHK_OBS).trigger(processingTime="10 seconds").start())

## 3. `lastProgress`: throughput y lag de Spark (3 muestras, una cada 20 s, tras 60 s de calentamiento)

In [6]:
time.sleep(60)
for muestra in range(3):
    p = consulta.lastProgress
    m = p["sources"][0].get("metrics", {}) if p else {}
    print(json.dumps({"muestra": muestra + 1, "hora": time.strftime("%H:%M:%S"), "service": "spark-streaming",
        "topic": TOPIC, "batchId": p["batchId"] if p else None,
        "numInputRows": p["numInputRows"] if p else None, "inputRowsPerSecond": p["inputRowsPerSecond"] if p else None,
        "processedRowsPerSecond": p["processedRowsPerSecond"] if p else None,
        "triggerExecutionMs": p["durationMs"].get("triggerExecution") if p else None,
        "avgOffsetsBehindLatest": m.get("avgOffsetsBehindLatest"), "maxOffsetsBehindLatest": m.get("maxOffsetsBehindLatest"),
        "status": "sin-datos" if not p else ("idle" if p["numInputRows"] == 0 else "processed")}))
    if muestra < 2: time.sleep(20)

{"muestra": 1, "hora": "01:11:57", "service": "spark-streaming", "topic": "viento-eventos", "batchId": 3, "numInputRows": 3, "inputRowsPerSecond": 0.3, "processedRowsPerSecond": 1.4, "triggerExecutionMs": 2151, "avgOffsetsBehindLatest": "0.0", "maxOffsetsBehindLatest": "0", "status": "processed"}


{"muestra": 2, "hora": "01:12:17", "service": "spark-streaming", "topic": "viento-eventos", "batchId": 4, "numInputRows": 3, "inputRowsPerSecond": 0.3, "processedRowsPerSecond": 1.4, "triggerExecutionMs": 2190, "avgOffsetsBehindLatest": "0.0", "maxOffsetsBehindLatest": "0", "status": "processed"}
{"muestra": 3, "hora": "01:12:37", "service": "spark-streaming", "topic": "viento-eventos", "batchId": 6, "numInputRows": 3, "inputRowsPerSecond": 0.3, "processedRowsPerSecond": 1.4, "triggerExecutionMs": 2209, "avgOffsetsBehindLatest": "0.0", "maxOffsetsBehindLatest": "0", "status": "processed"}


## 4. Prometheus y Grafana (capturas con reloj y usuario)

Consultas PromQL reales (pégalas en `localhost:49090` → *Graph*):
```
kafka_brokers
up{job="kafka-exporter"}
sum by (consumergroup) (kafka_consumergroup_lag{topic="viento-eventos"})
rate(kafka_topic_partition_current_offset{topic="viento-eventos"}[5m])
viento_stream_latencia_promedio_ms
viento_stream_validos_ratio
```
Targets esperados `UP`: `prometheus`, `kafka-exporter`, `pushgateway`.

La celda siguiente hace las mismas consultas contra la API de Prometheus y muestra los valores reales.

**Prueba de alerta:** `docker stop lambda26-kafka-exporter` → el panel *Exporter arriba* pasa a 0 → `docker start lambda26-kafka-exporter`.
**Prueba de lag:** detén `consumer_viento.py`, corre `python publicar_prueba.py rafaga 300` y mira subir el lag; vuelve a lanzar el consumer y mira bajarlo.

In [7]:
PROMETHEUS = os.getenv("PROMETHEUS", "http://prometheus:9090")
consultas = {
    "brokers Kafka": "kafka_brokers",
    "exporter arriba": 'up{job="kafka-exporter"}',
    "targets UP": "sum(up)",
    "lag por grupo (viento-eventos)": 'sum by (consumergroup) (kafka_consumergroup_lag{topic="viento-eventos"})',
    "mensajes/min viento-eventos": 'sum(rate(kafka_topic_partition_current_offset{topic="viento-eventos"}[5m])) * 60',
    "latencia promedio Spark (ms)": "viento_stream_latencia_promedio_ms",
    "ratio válidos Spark": "viento_stream_validos_ratio",
}
for nombre, q in consultas.items():
    try:
        res = requests.get(f"{PROMETHEUS}/api/v1/query", params={"query": q}, timeout=5).json()["data"]["result"]
        valores = [(r["metric"].get("consumergroup", ""), round(float(r["value"][1]), 3)) for r in res]
        print(f"{nombre:34s} {q:75s} → {valores if valores else 'sin datos aún'}")
    except Exception as ex:
        print(f"{nombre:34s} → error consultando Prometheus: {ex}")

brokers Kafka                      kafka_brokers                                                               → [('', 1.0)]
exporter arriba                    up{job="kafka-exporter"}                                                    → [('', 1.0)]
targets UP                         sum(up)                                                                     → [('', 3.0)]
lag por grupo (viento-eventos)     sum by (consumergroup) (kafka_consumergroup_lag{topic="viento-eventos"})    → [('uso-viento-group', 0.0), ('alertador-viento-group', 0.0)]
mensajes/min viento-eventos        sum(rate(kafka_topic_partition_current_offset{topic="viento-eventos"}[5m])) * 60 → [('', 11.369)]
latencia promedio Spark (ms)       viento_stream_latencia_promedio_ms                                          → [('', 154.0)]
ratio válidos Spark                viento_stream_validos_ratio                                                 → [('', 1.0)]


## 5. Latencia agregada por ventana de 1 minuto (sobre el Parquet persistido)

In [8]:
obs = spark.read.parquet(P_OBS)
print("eventos persistidos:", obs.count())
lat_min = (obs.groupBy(F.window("ts", "1 minute"))
           .agg(F.count("*").alias("eventos"), F.round(F.avg("latencyMs"), 1).alias("avgLatencyMs"),
                F.min("latencyMs").alias("minLatencyMs"), F.max("latencyMs").alias("maxLatencyMs"),
                F.round(F.avg(F.col("isValid").cast("int")), 3).alias("ratioValidos"))
           .orderBy("window"))
lat_min.show(30, truncate=False)
p50, p95 = obs.approxQuantile("latencyMs", [0.5, 0.95], 0.01)
print(f"Latencia publicación→procesado: p50 = {p50:.0f} ms | p95 = {p95:.0f} ms → umbral sugerido 'latencia sensible' = p95 × 2 = {2 * p95:.0f} ms")

eventos persistidos: 18
+------------------------------------------+-------+------------+------------+------------+------------+
|window                                    |eventos|avgLatencyMs|minLatencyMs|maxLatencyMs|ratioValidos|
+------------------------------------------+-------+------------+------------+------------+------------+
|{2026-11-02 20:11:00, 2026-11-02 20:12:00}|12     |5007.3      |1518        |8503        |1.0         |
|{2026-11-02 20:12:00, 2026-11-02 20:13:00}|6      |2985.0      |154         |5816        |1.0         |
+------------------------------------------+-------+------------+------------+------------+------------+

Latencia publicación→procesado: p50 = 2870 ms | p95 = 8503 ms → umbral sugerido 'latencia sensible' = p95 × 2 = 17006 ms


## 6. Alertas propuestas (justificar con los valores medidos arriba)
| Alerta | Regla | Justificación con mis datos |
|---|---|---|
| Kafka no visible | `kafka_brokers < 1` | sin broker no hay pipeline |
| Exporter caído | `up{job="kafka-exporter"} == 0` | quedamos ciegos al lag |
| Lag sostenido | `sum(kafka_consumergroup_lag{topic="viento-eventos"}) > 100` por 5 min | en operación normal llegan ~3 ev/min: 100 de lag = >30 min de atraso |
| Latencia sensible | `viento_stream_latencia_promedio_ms > (p95 × 2)` | usar el valor impreso arriba: el doble de lo normal medido |
| Datos viejos | `viento_stream_edad_dato_ms > 1800000` (30 min) | el modelo actualiza cada 15 min: 2 pasos sin cambio = API trabada |
| Calidad | `viento_stream_validos_ratio < 0.95` | cambio de contrato o de la fuente |

## 7. Plantilla de costos (medida con datos propios)

In [9]:
pico = lat_min.agg(F.max("eventos")).first()[0] or 0
eventos_seg_pico = pico / 60
tam_bytes = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(P_OBS) for f in fs if f.endswith(".parquet"))
horas_medidas = max((obs.agg((F.max("timestamp") - F.min("timestamp")) / 3_600_000).first()[0] or 0), 1 / 60)
gb_mes = tam_bytes / horas_medidas * 730 / 1e9

# Precios de referencia (VERIFICAR en la web de AWS antes de entregar)
PRECIO_T3_MEDIUM_H = 0.0416     # USD/h, on-demand, us-east-1
PRECIO_S3_GB_MES = 0.023        # USD/GB-mes, S3 Standard
componentes = {"broker Kafka + exporter": 1, "Spark streaming (driver local)": 1, "obs (Prometheus+Grafana+Pushgateway)": 1}
computo = len(componentes) * PRECIO_T3_MEDIUM_H * 730
almacen = gb_mes * PRECIO_S3_GB_MES
print(f"Pico medido: {pico} eventos/min = {eventos_seg_pico:.3f} ev/s")
print(f"Crecimiento Parquet: {tam_bytes/1e6:.3f} MB en {horas_medidas:.2f} h → {gb_mes:.4f} GB/mes")
print(f"Cómputo: {len(componentes)} × t3.medium × 730 h = US$ {computo:.2f}/mes")
print(f"Almacenamiento S3: US$ {almacen:.4f}/mes   | TOTAL ≈ US$ {computo + almacen:.2f}/mes (sin tráfico de red)")
for factor in (10, 100):
    print(f"  ×{factor} estaciones → {eventos_seg_pico*factor:.2f} ev/s, {gb_mes*factor:.3f} GB/mes")

Pico medido: 12 eventos/min = 0.200 ev/s
Crecimiento Parquet: 0.122 MB en 0.02 h → 4.0952 GB/mes
Cómputo: 3 × t3.medium × 730 h = US$ 91.10/mes
Almacenamiento S3: US$ 0.0942/mes   | TOTAL ≈ US$ 91.20/mes (sin tráfico de red)
  ×10 estaciones → 2.00 ev/s, 40.952 GB/mes
  ×100 estaciones → 20.00 ev/s, 409.522 GB/mes


**Plan de escalado (completar con tus números):**
1. Si el lag pasa 100 de forma sostenida o `triggerExecution` > trigger → primero **más particiones** en `viento-eventos` + más cores al Spark (el broker con 3 ev/min está ocioso).
2. Si el Parquet supera ~1 GB/mes → compactar archivos pequeños por día (el streaming genera muchos archivos chicos) y mover a S3 IA.
3. Si se duplica el volumen: escala **Spark** primero (CPU de parseo/validación), el broker después.

## 8. Dejar el stream corriendo para Grafana
El stream de la sección 2 **sigue corriendo** y empujando métricas al Pushgateway cada 10 s.
Abre Grafana (`localhost:43000` → *Dashboards* → *Viento Juliaca — Pipeline U2*) y toma las capturas.
Cuando termines, corre la celda de abajo (sin comentar) para detenerlo.

In [11]:
consulta.stop(); print("stream detenido")

stream detenido
